##### data engineering notebooks

In [23]:
## include all required libraries here

import os
from copy import deepcopy
import glob
import pathlib
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely as shpy
from shapely import geometry as shpy_Geom
import reverse_geocoder as rg
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns

## import local lib
# import api
import api.utils

In [24]:
PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_DIRPATH = PROJECT_ROOT / 'data' / 'raw'

GEO_DATA_DIRPATH = PROJECT_ROOT / 'data' / 'supporting_dataset' / 'US_State_geo'

shpfile_path = GEO_DATA_DIRPATH / 'cb_2016_us_state_5m.shp'

pop_filepath = os.path.join( GEO_DATA_DIRPATH, 'us_pop_by_state.csv' )



WindowsPath('e:/Learning_course/MLOps/coursera_packt/greenHouse_Emission_predictor/data/supporting_dataset/US_State_geo')

In [25]:
dir_ls = glob.glob(  os.path.join( DATA_DIRPATH, '**/' ), recursive= True  )[1:]  ## [1:]  to avoid the very root dir

In [26]:
## get all files from sub-directories

emission_dictn = dict()

for edir in dir_ls:

    ## inside each sector dir
    file_ls = [  str(efile) for efile in pathlib.Path(edir).rglob( '*.csv' ) ]
    emissionFile_ls = [  efile for efile in file_ls if efile.endswith('_emissions-sources.csv')  ]
    
    sectornm = os.path.basename( os.path.normpath(edir) )
    sectornm = sectornm[:5] + '_df'  ## example: agric_df, fores_df

    concat_df = pd.DataFrame()

    for eEmission in emissionFile_ls:
        _dfi = pd.read_csv( eEmission )
        subSector = os.path.basename(eEmission).replace( '_emissions-sources.csv', '' )   ## clean subsector, ex. manure-management-cattle-feedlot
        _dfi['emission_subsector'] = subSector  ## a col value with subsector for later identification about the subsection source file
        concat_df = pd.concat(  [ concat_df, _dfi ], ignore_index= True )
    
    emission_dictn[sectornm] = concat_df  ## --> put all the section dfs into dict

C:\Users\madhur\AppData\Local\Temp\ipykernel_33940\3584533552.py:17: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  _dfi = pd.read_csv( eEmission )


In [ ]:
_usa_gdf = gpd.read_file( shpfile_path )
_uspop_df = pd.read_csv( pop_filepath )


In [27]:
# ## checking exhuastive list of unique gas type in all sections
# for k, v_df in emission_dictn.items():
#     try: print(  sorted( v_df['gas'].unique() )   )
#     except: pass

In [28]:
## replace: 'co2', 'co2e_100yr', 'co2e_20yr'  to 'co2' for pragmatic purpose
## for a dataframe
# agric_df['gas'] = agric_df['gas'].replace( {'co2e_100yr': 'co2', 'co2e_20yr': 'co2'}, regex= True )

for k, v_df in emission_dictn.items():
    try:
        v_df['gas'] = v_df['gas'].replace( {'co2e_100yr': 'co2', 'co2e_20yr': 'co2'}, regex= True )
        print(  sorted( v_df['gas'].unique() )  )
    except: pass

['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']
['ch4', 'co2', 'n2o']


In [29]:
emission_dictn.keys()

dict_keys(['agric_df', 'fores_df', 'fossi_df', 'manuf_df', 'miner_df', 'power_df', 'trans_df', 'waste_df'])

In [30]:
## get summary of the emissions_quantity for each gas type

emiSummary_df = pd.DataFrame()

for k, v_df in emission_dictn.items():
    try:
        v_df['gas'] = v_df['gas'].astype( 'category' )

        dfi = v_df[ ['gas', 'emissions_quantity'] ].groupby( 'gas' ).sum().reset_index( drop= False )
        dfi['sector'] = k.split('_')[0]

        emiSummary_df = pd.concat(  [ emiSummary_df, dfi ], ignore_index= True  )
    except: pass

C:\Users\madhur\AppData\Local\Temp\ipykernel_33940\3341928170.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dfi = v_df[ ['gas', 'emissions_quantity'] ].groupby( 'gas' ).sum().reset_index( drop= False )
C:\Users\madhur\AppData\Local\Temp\ipykernel_33940\3341928170.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dfi = v_df[ ['gas', 'emissions_quantity'] ].groupby( 'gas' ).sum().reset_index( drop= False )
C:\Users\madhur\AppData\Local\Temp\ipykernel_33940\3341928170.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=

In [31]:
### for this project, to make it manageable only workign with Power dataset
_power_df = deepcopy( emission_dictn['power_df'] )

In [32]:
## clean up the data, remove unnecessary cols, rename some cols, explode 'source_type' col, convert datatypes, create geometry col

power_df = (   deepcopy( _power_df )
    .drop( columns= [ 'iso3_country', 'original_inventory_sector', 'temporal_granularity', 'capacity_units', 'activity_units', 'created_date', 'source_name', 'other1', 'other1_def', 'emission_subsector', 'emissions_factor_units'  ] )
    .rename(  columns= {  'other2': 'biomass_emissions', 'other3': 'biomass_capacity', 'other4': 'biomass_generation' }  )
    .drop( columns= [ 'other2_def', 'other3_def', 'other4_def'  ] )

    ## explode based on 'source_type' col
    .assign(  source_type = lambda df: df['source_type'].str.split(',')  )
    .explode(  'source_type', ignore_index= True  )
    .assign( source_type = lambda df: df['source_type'].astype('category') )
    .reset_index( drop= True )

    .assign(
        ## convert object into datetime
        start_time = lambda df: pd.to_datetime( df['start_time'] ),
        end_time = lambda df: pd.to_datetime( df['end_time'] ),
        # clean 'source_type' column
        source_type = lambda df: df['source_type'].str.strip().str.lower(),
    )

    ## remove alaska and hawaai for simplicity
    .query( 'lon > -130' )

    .dropna( subset= ['emissions_quantity'] )
    ## get a geometry column using lat lon
    .assign(    geometry = lambda df: df.apply(  lambda _df: shpy.Point( _df['lon'], _df['lat'] ), axis= 1  )    )
)



print(  power_df.shape  ); power_df.head()

(47370, 18)


,source_id,start_time,end_time,gas,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,modified_date,source_type,lat,lon,biomass_emissions,biomass_capacity,biomass_generation,geometry_ref,geometry
0,25448848,2021-01-01,2021-12-31,n2o,NaN,NaN,138,0.324,392000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.9708 34.0128)
1,25448848,2022-01-01,2022-12-31,co2,214000.0,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.9708 34.0128)
2,25448848,2022-01-01,2022-12-31,co2,214000.0,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.9708 34.0128)
3,25448848,2022-01-01,2022-12-31,n2o,NaN,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.9708 34.0128)
4,25448848,2022-01-01,2022-12-31,ch4,NaN,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.9708 34.0128)


In [33]:
## convert pandas into geopandas so that it have goespatial info
power_gpd = gpd.GeoDataFrame( power_df.copy(), geometry= 'geometry', crs= 'EPSG:2264' )
## plotting only CO2
powerCo2_gpd = ( power_gpd.copy()
    .query( 'gas == "co2"' )
    .drop( columns= 'gas' )
    .assign(  emi_log = lambda df: np.log2( df['emissions_quantity'] )  )
    [  lambda df: df['emi_log'] != -np.inf  ]  ## remove -np.inf from emi_log
)
print(  powerCo2_gpd.shape  )
powerCo2_gpd.head()

(26637, 18)


e:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


,source_id,start_time,end_time,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,modified_date,source_type,lat,lon,biomass_emissions,biomass_capacity,biomass_generation,geometry_ref,geometry,emi_log
1,25448848,2022-01-01,2022-12-31,214000.0,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.971 34.013),17.707251
2,25448848,2022-01-01,2022-12-31,214000.0,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.971 34.013),17.707251
5,25448848,2022-01-01,2022-12-31,214000.0,0.513,138,0.345,417000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.971 34.013),17.707251
6,25448848,2021-01-01,2021-12-31,201000.0,NaN,138,0.324,392000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.971 34.013),17.616836
7,25448848,2021-01-01,2021-12-31,201000.0,NaN,138,0.324,392000,2023-11-01 10:00:00,gas,34.0128,-85.9708,NaN,NaN,NaN,trace_-85.9708_34.0128,POINT (-85.971 34.013),17.616836


In [34]:
### removing alask and other unrequired geographies for our case
CLIP_BOX = shpy.geometry.box( -150, 25, -60, 50 )  ## xmin, ymin, xmax, ymax (longitude, latitude)
state_san_ls = [  'Alaska', 'American Samoa', 'Guam', 'Hawaii', 'Puerto Rico', 'United States Virgin Islands', 'Commonwealth of the Northern Mariana Islands' ]



usa_gdf = (   deepcopy( _usa_gdf )
    [  lambda df:  ~df['NAME'].isin( state_san_ls )  ]
    ## remove clip the polygon outside clip_box for aesthetic purpose
    .assign(   geometry = lambda df: df['geometry'].intersection( CLIP_BOX )  )
)
usa_gdf.head(1)

,STATEFP,STATENS,AFFGEOID,GEOID,STUSPS,NAME,LSAD,ALAND,AWATER,geometry
0,01,01779775,0400000US01,01,AL,Alabama,00,131173688951,4593686489,"MULTIPOLYGON (((-88.03661 30.52049, -88.02473 ..."


In [ ]:
## get population data by states


uspop_df = (  deepcopy( _uspop_df )
    .drop( columns= [ 'rank', 'percent_of_total']  )
    .replace(  { 'DC' : 'District of Columbia'  }  )
)

<div class= 'alert alert-block alert-success'>
<b>Python Geospatial library</b>
<p>
Adopting Geopandas for including spatial info
</div>

In [36]:
## Spatial Join to get total emission per state

powerUSA_gdf = (  deepcopy(powerCo2_gpd)
    .filter(  [ 'source_id', 'emissions_quantity', 'geometry', 'activity' ], axis= 'columns'  )
    ## spatial join to get which point lies within which state polygon
    .sjoin( usa_gdf[ ['STUSPS', 'NAME', 'geometry'] ], how= 'inner', predicate= 'within' )
    
    .groupby( 'NAME' )[['emissions_quantity']].sum().reset_index()  ## groupby for summerization of emissions_quantity
    .merge(  usa_gdf[['STUSPS', 'NAME', 'geometry']], how= 'right', on= 'NAME'  )
    .pipe(  lambda df: gpd.GeoDataFrame( df, geometry= 'geometry', crs= usa_gdf.crs )  )

    ## join with population data 
    .merge( uspop_df, how= 'left', left_on= 'NAME', right_on= 'state' )
    .drop(  columns= [ 'STUSPS', 'NAME' ]  )
    .rename(  columns= { '2020_census': 'pop2020' }  )

    ## normalizing the emission based on state area
    .assign(
        area = lambda df: df.area,
        ## normalized emission amount per area
        emission_AreaNorm = lambda df: df.apply( lambda _df: _df['emissions_quantity']/_df['area'], axis= 1 ),
        ## normalized emission amount per population ( emission per capita)
        emission_PopNorm = lambda df: df.apply( lambda _df: _df['emissions_quantity']/_df['pop2020'], axis= 1 )
    )
    
)

e:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\.venv\Lib\site-packages\geopandas\geodataframe.py:2573: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:2264
Right CRS: EPSG:4269

  return geopandas.sjoin(
C:\Users\madhur\AppData\Local\Temp\ipykernel_33940\3057511220.py:19: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area = lambda df: df.area,


In [45]:
powerCo2_df = (  deepcopy( powerCo2_gpd )
    # .drop(  columns= [ 'geometry', 'geometry_ref' ]  )
    .drop(  columns= [ 'emi_log', 'source_id','lat', 'lon' ]  )
    .assign(   source_type = lambda df: df['source_type'].astype('category')  )

    ## spatial join with powerUSA_gdf to get state area and pop
    .sjoin(  powerUSA_gdf[ ['state', 'geometry', 'area', 'pop2020' ] ], how= 'inner', predicate= 'within'  )
    .drop(  columns= [ 'geometry', 'geometry_ref', 'index_right' ], errors= 'ignore' )
    .reset_index( drop= True )
)

powerCo2_df.head(2)

e:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\.venv\Lib\site-packages\geopandas\geodataframe.py:2573: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:2264
Right CRS: EPSG:4269

  return geopandas.sjoin(


,start_time,end_time,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,modified_date,source_type,biomass_emissions,biomass_capacity,biomass_generation,state,area,pop2020
0,2022-01-01,2022-12-31,214000.0,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,NaN,NaN,NaN,Alabama,12.899239,5024279
1,2022-01-01,2022-12-31,214000.0,NaN,138,0.345,417000,2023-11-01 10:00:00,gas,NaN,NaN,NaN,Alabama,12.899239,5024279


In [38]:
# Hard-code datatypes

DTYPE_MAP: dict = {
    'start_time': 'datetime',
    'end_time': 'datetime',
    'emissions_quantity': 'float',
    'emissions_factor': 'float',
    'capacity': 'float',
    'capacity_factor': 'float',
    'activity': 'float',
    'modified_date': 'datetime',
    'source_type': 'category',
    'biomass_emissions': 'float',
    'biomass_capacity': 'float',
    'biomass_generation': 'float',
    'state': 'category',
    'area': 'float',
    'pop2020': 'float'
}

powerCo2_df = api.utils.enforce_dtypes( powerCo2_df, DTYPE_MAP )

print('\nApplied dtypes:')
print( powerCo2_df[ [c for c in DTYPE_MAP.keys() if c in powerCo2_df.columns] ].dtypes )


Applied dtypes:
start_time            datetime64[ns]
end_time              datetime64[ns]
emissions_quantity           float64
emissions_factor             float64
capacity                     float64
capacity_factor              float64
activity                     float64
modified_date         datetime64[ns]
source_type                 category
biomass_emissions            float64
biomass_capacity             float64
biomass_generation           float64
state                       category
area                         float64
pop2020                      float64
dtype: object


In [40]:
miss_df = (
    pd.DataFrame( powerCo2_df.isna().sum() , columns= [ 'miss_count'] )
    .reset_index().rename( columns= { 'index': 'colnm' } )
    .query( 'miss_count > 0'  ).reset_index( drop= True  )
)


print(miss_df)

                colnm  miss_count
0    emissions_factor       17698
1   biomass_emissions       26073
2    biomass_capacity       26073
3  biomass_generation       26073


In [41]:
## 30 percent threshold of total rows
miss_30_threshold = powerCo2_df.shape[0]*30/100

miss_col_less_threshold = miss_df.query( f'miss_count < {miss_30_threshold}' )
miss_col_more_threshold = miss_df.query( f'miss_count >= {miss_30_threshold}' )
# ['colnm'].tolist()
miss_col_more_threshold

,colnm,miss_count
0,emissions_factor,17698
1,biomass_emissions,26073
2,biomass_capacity,26073
3,biomass_generation,26073


In [42]:
# drop col which have more than 30% missing values
powerCo2_df = powerCo2_df.drop( columns= miss_col_more_threshold['colnm'].tolist() )
# impute col which have less than 30% missing values
col2Impute = miss_col_less_threshold['colnm'].to_list()
## get key value pairs of DTYPE_MAP for elements in  col2Impute
impute_dtype_map = {  k: v for k, v in DTYPE_MAP.items() if k in col2Impute  }

powerCo2_df = api.utils.impute_by_dtype(powerCo2_df, impute_dtype_map)

# show top rows and missing counts after imputation
print('\nMissing counts after imputation:')
print(  powerCo2_df.isna().sum().loc[ impute_dtype_map.keys() ]  )


Imputation summary (col, action, missing_before, missing_after):

Missing counts after imputation:
Series([], dtype: int64)


In [43]:
## confirm that no negative values in numeric columns
num_col_ls = [ 'emissions_quantity', 'capacity',
    'capacity_factor', 'activity', 'area', 'pop2020' ]
assert (  powerCo2_df[  num_col_ls  ] >= 0  ).all().all()

In [44]:
print( '\nData Engineering complete!' )


Data Engineering complete!
